# nVision Python port — notebook workspace

This notebook is the development and validation workspace for a license-free Python implementation of the isolated nVision core. The MATLAB implementation remains the reference. Execute cells manually and validate on the Day 1 ECoG before packaging for Linux.

Scope: `.dat` float32 input, `winnyAll.m`, required filters/features, five-feature `net_219`, MATLAB-compatible normalization, network inference, `find_seizures.m`, and result export.

## Configuration

Update paths if this notebook is moved. No files are written by the notebook until an explicit export cell is run.

In [1]:
from pathlib import Path
import math
import json
import numpy as np
from scipy import io, signal, stats

CORE = Path(r'C:/Users/mstrh/Downloads/Work/nVision Linux port/nVision_core')
ECOG = CORE / 'test ecogs' / 'Day 1-20231009Sel_Merged_h00hz_.dat'
NETWORK = CORE / 'Networks' / 'net_219.mat'
NETWORK_NUMERIC = CORE / 'python_port' / 'net_219_numeric.mat'
FILTER_HP_200 = CORE / 'HP_fs200_fc2_order200.mat'
OUTPUT = CORE / 'python_port' / 'outputs'
RATE = 200
THRESHOLD = 1.2
USE_FEATS_5 = np.array([1, 0, 1, 1] + [0] * 16 + [1, 1] + [0] * 5, dtype=bool)
FEATURE_SPOTS = np.array([0, 2, 3, 20, 21])  # MATLAB [1 3 4 21 22], zero-based here


## MATLAB-compatible primitives

In [2]:
def read_dat_float32(path: Path) -> np.ndarray:
    return np.fromfile(path, dtype='<f4')

def matlab_kurtosis(x):
    return float(stats.kurtosis(np.asarray(x), fisher=False, bias=False))

def matlab_skewness(x):
    return float(stats.skew(np.asarray(x), bias=False))

def mapminmax_fit(x, ymin=-1.0, ymax=1.0):
    x = np.asarray(x, dtype=float)
    xmin = np.min(x, axis=1, keepdims=True)
    xmax = np.max(x, axis=1, keepdims=True)
    gain = (ymax - ymin) / np.where(xmax == xmin, 1.0, xmax - xmin)
    return (x - xmin) * gain + ymin, {'xmin': xmin, 'xmax': xmax, 'gain': gain, 'ymin': ymin, 'ymax': ymax}

def mapminmax_apply(x, settings):
    x = np.asarray(x, dtype=float)
    return (x - settings['xmin']) * settings['gain'] + settings['ymin']

def fft_band_fraction(data, rate, low, high):
    n = len(data)
    nfft = 2 ** int(np.ceil(np.log2(n)))
    freqs = np.linspace(0, rate / 2, nfft // 2 + 1)
    power = np.abs(np.fft.fft(data, nfft) / n)[:nfft // 2 + 1] ** 2
    start = np.flatnonzero(freqs > low)[0]
    end = np.flatnonzero(freqs > high)[0]
    return float(power[start:end].sum() / power.sum())


## Feature extraction and `winnyAll.m`

In [3]:
def parameters_5_vs_26(data, rate, use_feats):
    data = np.asarray(data, dtype=float)
    if use_feats.sum() == 5:
        n = len(data)
        values = [np.mean(data ** 2), np.std(data, ddof=1), np.abs(np.diff(data)).sum(),
                  fft_band_fraction(data, rate, 12, 20), fft_band_fraction(data, rate, 20, 40)]
        return np.asarray(values, dtype=float)
    if use_feats.sum() != 5:
        raise ValueError('This notebook currently supports the five-feature net_219 path only.')

def load_hp_filter(rate=RATE):
    if rate != 200:
        raise NotImplementedError('Only the validated 200 Hz path is enabled currently.')
    return np.asarray(io.loadmat(FILTER_HP_200)['HP_fs200_fc2_order200']).squeeze().astype(float)

def matlab_buffer_half_overlap(data, window):
    # MATLAB buffer(data, window, window*0.5): columns are 50% overlapped.
    step = window // 2
    count = int(np.ceil(len(data) / step))
    columns = []
    for i in range(count):
        column = data[i * step:i * step + window]
        if len(column) < window:
            column = np.pad(column, (0, window - len(column)), constant_values=0)
        columns.append(column)
    return np.column_stack(columns)

def winny_all(data, rate, threshold, hp_b, lp_b=None, use_feats=USE_FEATS_5):
    data = np.asarray(data, dtype=float).copy()
    if rate == 200:
        noise = np.isnan(data)
        data -= np.nanmean(data)
        data[noise] = 0
    elif rate in (500, 1000):
        noise = data == 100
        data[noise] = np.nan
        data -= np.nanmean(data)
        data[noise] = 0
    if rate == 1000:
        data = signal.filtfilt(lp_b, [1.0], data)[::2]
        noise = noise[::2]
        rate = 500
    data = signal.filtfilt(hp_b, [1.0], data, method='pad', padtype='odd')
    data[noise] = 100
    windows = matlab_buffer_half_overlap(data, rate * 2)
    rows = []
    for window in windows.T:
        if np.mean(np.abs(window) > threshold) or np.std(window) == 0:
            rows.append(np.full(use_feats.sum(), np.nan))
        else:
            rows.append(parameters_5_vs_26(window, rate, use_feats))
    return np.asarray(rows)

def process_feature_chunks(raw, rate=RATE, threshold=THRESHOLD, use_feats=USE_FEATS_5):
    chunk_size = 3600 * rate
    hp = load_hp_filter(rate)
    chunks = [raw[start:start + chunk_size] for start in range(0, len(raw), chunk_size)]
    return np.vstack([winny_all(chunk, rate, threshold, hp, use_feats=use_feats) for chunk in chunks])


## Network adapter

`net_219` is a legacy MATLAB feed-forward network: 5 inputs → 50 `tansig` hidden units → 1 `purelin` output. The numeric weights and stored second-stage `mapminmax` ranges are exported in `net_219_numeric.mat`; this is an inference export, not a retrained or modified network.

In [4]:
def load_net219_numeric(path=NETWORK_NUMERIC):
    m = io.loadmat(path, squeeze_me=True)
    weights = {
        'IW': np.asarray(m['IW'], dtype=float).reshape(50, 5),
        'LW': np.asarray(m['LW'], dtype=float).reshape(1, 50),
        'b1': np.asarray(m['b1'], dtype=float).reshape(50, 1),
        'b2': np.asarray(m['b2'], dtype=float).reshape(1, 1),
        'xmin': np.asarray(m['xmin'], dtype=float).reshape(5),
        'xmax': np.asarray(m['xmax'], dtype=float).reshape(5),
    }
    return weights, float(m['ymin']), float(m['ymax'])

def mapminmax_fixed(x, xmin, xmax, ymin=-1.0, ymax=1.0):
    x = np.asarray(x, dtype=float)
    scale = (ymax - ymin) / np.where(xmax == xmin, 1.0, xmax - xmin)
    return (x - xmin) * scale + ymin

def forward_net_219(features, model):
    weights, ymin, ymax = model
    x = np.asarray(features, dtype=float)
    if x.ndim == 1:
        x = x[:, None]
    x = mapminmax_fixed(x, weights['xmin'][:, None], weights['xmax'][:, None], ymin, ymax)
    hidden = np.tanh(weights['IW'] @ x + weights['b1'])
    return weights['LW'] @ hidden + weights['b2']

# Manual load; leave execution to the validation stage.
# net219 = load_net219_numeric()

def classify_feature_rows(feature_rows, quiet_settings, net219):
    feature_rows = np.asarray(feature_rows, dtype=float)
    mapped = mapminmax_apply(feature_rows.T, quiet_settings)
    selected = mapped if mapped.shape[0] == 5 else mapped[FEATURE_SPOTS, :]
    return forward_net_219(selected, net219).reshape(-1)


## `find_seizures.m` translation

In [5]:
def find_seizures(outputs, sensitivity=1.0, duration=6, interval=1, classification_mode=(1, 0)):
    results = []
    for values in outputs:
        values = np.asarray(values, dtype=float).reshape(-1)
        if classification_mode[0]:
            hits = np.convolve(np.nan_to_num(values > sensitivity, nan=False).astype(int),
                               np.ones(duration, dtype=int), mode='valid') >= duration
            candidate = np.flatnonzero(hits) + duration - 1
        else:
            candidate = np.array([], dtype=int)
        accepted = []
        for index in candidate:
            if not accepted or index > accepted[-1] + interval * 60:
                accepted.append(int(index))
        results.append(np.asarray(accepted, dtype=int))
    return results

def export_result_json(path, event_starts, network_output, feature_rows, sample_rate, threshold, quiet_seconds):
    payload = {
        'format': 'nVision-python-result-v1',
        'sample_rate_hz': int(sample_rate),
        'noise_threshold': float(threshold),
        'quiet_reference_seconds': float(quiet_seconds),
        'feature_names': ['mean_square', 'std', 'absolute_difference_sum', 'power_fraction_12_20hz', 'power_fraction_20_40hz'],
        'feature_shape': list(np.asarray(feature_rows).shape),
        'network_output': np.asarray(network_output, dtype=float).reshape(-1).tolist(),
        'events': [{'start_epoch': int(i), 'start_seconds': int(i)} for i in np.asarray(event_starts).reshape(-1)],
    }
    Path(path).write_text(json.dumps(payload, indent=2), encoding='utf-8')
    return Path(path)


## End-to-end execution (run manually)

After loading `net219`, process the ECoG in 3,600-second chunks, classify each feature row, call `find_seizures`, and export a result bundle. Compare the event timeline with the MATLAB result before moving to Linux packaging.

In [6]:
OUTPUT.mkdir(parents=True, exist_ok=True)

raw = read_dat_float32(ECOG)
hp = load_hp_filter(RATE)
features = process_feature_chunks(raw, RATE, THRESHOLD, USE_FEATS_5)

quiet = raw[:60 * RATE]
quiet_features = winny_all(quiet, RATE, THRESHOLD, hp, use_feats=USE_FEATS_5)
_, q_settings = mapminmax_fit(quiet_features.T)
net219 = load_net219_numeric()
network_outputs = classify_feature_rows(features, q_settings, net219)
events = find_seizures([network_outputs])
result_path = export_result_json(OUTPUT / 'day1_python_result.json', events[0], network_outputs, features, RATE, THRESHOLD, 60)
print(result_path)
print('Detected events:', len(events[0]))

C:\Users\mstrh\Downloads\Work\nVision Linux port\nVision_core\python_port\outputs\day1_python_result.json
Detected events: 0
